# 05-3 트리의 앙상블

## 3️⃣ 랜덤 포레스트 (Random Forest)

In [33]:
import numpy as np
import pandas as pd

#와인데이터 불러오기
wine = pd.read_csv('./data/wine.csv')
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [34]:
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

In [35]:
#훈련 세트와 테스트세트 나누기
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42)

In [36]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=1, random_state=42)

#교차 검증
score = cross_validate(rf, train_input, train_target, return_train_score=True, n_jobs=1)
score

{'fit_time': array([0.3055408 , 0.2810154 , 0.34812856, 0.27836585, 0.2947681 ]),
 'score_time': array([0.01565409, 0.01562166, 0.03204536, 0.02577662, 0.00908208]),
 'test_score': array([0.88461538, 0.88942308, 0.90279115, 0.88931665, 0.88642926]),
 'train_score': array([0.9971133 , 0.99663219, 0.9978355 , 0.9973545 , 0.9978355 ])}

In [37]:
#훈련세트 점수
np.mean(score['train_score'])

np.float64(0.9973541965122431)

In [38]:
#테스트 세트 점수
np.mean(score['test_score'])

np.float64(0.8905151032797809)

In [39]:
#학습
rf.fit(train_input, train_target)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [40]:
#특성 중요도
rf.feature_importances_

array([0.23167441, 0.50039841, 0.26792718])

In [41]:
# [0.23101011 0.52164748 0.24734241]

### 자체적으로 모델을 평가하는 기능이 있다.

In [42]:
rf = RandomForestClassifier(oob_score=True, n_jobs=1, random_state=42)
rf.fit(train_input, train_target)

# 랜덤 포레스트는 각 트리를 만들때 Bootstrap Sampling을 사용한다.
# 즉 훈련데이터에서 복원추출을 한다. => 그래서 선택되지 않은 데이터가 항상 남는다.
# 이 데이터를 Out Of Bag (OOB)라고 하고
# 각 트리마다 훈련에 사용되지 않은 OOB 데이터로 그 트리를 평가합니다

# OOB 데이터는 부트스트랩 샘플링에서 선택되지 않은 데이터이며,
# 이 데이터를 사용하여 해당 트리를 평가하고 전체 OOB score를 계산한다.

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [43]:
rf.oob_score_

0.8934000384837406

# 4️⃣ 엑스트라 트리 (Extra Trees)

In [44]:
# - 차이점 : 
# 랜덤포레스트 - 최적의 분할 찾기, 
# 엑스트라트리 - 각 결정트리를 만들 때 전체 훈련세트를 사용함, 랜덤으로 분할, 즉 더 많은 랜덤성을 갖는다.

In [45]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(random_state=42)
score = cross_validate(et, train_input, train_target, return_train_score=True)
score


{'fit_time': array([0.26608896, 0.23716211, 0.25032544, 0.25011849, 0.24937892]),
 'score_time': array([0.03382087, 0.03340006, 0.01562834, 0.02277756, 0.03078771]),
 'test_score': array([0.88365385, 0.87884615, 0.90375361, 0.88835419, 0.88931665]),
 'train_score': array([0.9971133 , 0.99663219, 0.998076  , 0.997595  , 0.9978355 ])}

In [46]:
#훈련세트 점수(검증)
np.mean(score['test_score'])

np.float64(0.8887848893166506)

In [47]:
#테스트 점수

In [48]:
#엑스트라 트리도 특성 중요도를 제공한다
et.fit(train_input,train_target)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=FalseWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",False
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metr

In [49]:
#특성중요도
et.feature_importances_

array([0.20183568, 0.52242907, 0.27573525])

# 그레디언트 부스팅(gradient boosting)

In [50]:
#이전 모델의 오차을 다음 모델이 보완하는 방법
#-사이킷런의 gradientBoostingClassifier는 기본적으로 깊이가 3인 결정 트리를 100개 사용한다.
#깊이가 얕은 결정트리를 사용하기 때문에 과대적합에 강하고 일반적으로 높은 일반화 성능을 기대할 수 있다.

In [51]:
from sklearn.ensemble import GradientBoostingClassifier
gb=GradientBoostingClassifier(random_state=42)
score=cross_validate(gb,train_input, train_target,return_train_score=True, n_jobs=1)
score

{'fit_time': array([0.33983731, 0.27001071, 0.25860977, 0.26683688, 0.26439381]),
 'score_time': array([0.01500177, 0.00201082, 0.        , 0.00201035, 0.        ]),
 'test_score': array([0.86634615, 0.87019231, 0.89412897, 0.86044273, 0.86910491]),
 'train_score': array([0.89006495, 0.88958383, 0.88239538, 0.89249639, 0.88600289])}

In [52]:
#훈련세트점수
np.mean(score['train_score'])

np.float64(0.8881086892152563)

In [53]:
#테스트 점수(검증데이터 정확동)
np.mean(score['test_score'])

np.float64(0.8720430147331015)

In [54]:
#과대적합 되지 않으면서 좋은 결과가 나옴,
#그레이디언트 부스팅은 결정 트리의 개수를 늘려도 과대적합에 매우 강함
#결정트리를 늘려보자.

In [ ]:
#결정트리를 500으로 늘림
gb=GradientBoostingClassifier(n_estimators=500,random_state=42, learning_rate=0.5)
scores=cross_validate(gb,train_input,train_target,return_train_score=True)
np.mean(scores['train_score'])

#learning_rate는 새로 추가되는 트리가 이전 모델을 얼마나 강하게 수정할지 결정하는 값
#learning_rate=0.5, 학습속도빠름, 오류를 강하게 수정
#learning_rate=0.1 , 학습 안정적, 조금씩 수정 
#머신러닝에서는 거의 공식처럼 이렇게 ㅆ브니다. 
#n_estimators =100~500, learning_rate=0.1
#n_estimators =500~1000, learing_rate=0.05
#n_estimators =1000개이상 learing_rate=0.01
#기존모델+새트리*learing_rate

np.float64(0.9612293710441413)

In [60]:
np.mean(scores['test_score'])

np.float64(0.8770435700007404)

In [61]:
gb.fit(train_input,train_target)
gb.feature_importances_

array([0.16797208, 0.67238031, 0.15964761])

# 히스토그램 기반 그레이디언트 부스팅


In [63]:
from sklearn.ensemble import HistGradientBoostingClassifier
hgb=HistGradientBoostingClassifier(random_state=42)
scores=cross_validate(hgb, train_input,train_target,return_train_score=True, n_jobs=1)
np.mean(scores['train_score']), np.mean(scores['test_score'])

(np.float64(0.9321723946453317), np.float64(0.8801241948619236))

In [70]:
from sklearn.inspection import permutation_importance

hgb.fit(train_input, train_target)

result = permutation_importance(
    hgb,
    train_input,
    train_target,
    n_repeats=10,
    random_state=42
)

print('특성중요도', result.importances)
print('특성중요도 평균', result.importances_mean)
print('표준편차', result.importances_std)

#pernutation_importance: 하나의 특성을 무작위로 섞어서 모델 성능이 얼마나 떨어지는지를 본다.
#성능이 많이 떨어지면-> 중요한 특성
#성능이 거의 안 떨어지면-. 덜 중요한 특성
# alcohol, suger, ph
#result.importances:(특성개수, 반복횟수)
#result.importances_mean: 각 특성의 평균 중요도
#result.importances_std: 중요도의 변동 정도

특성중요도 [[0.08793535 0.08350972 0.08908986 0.08312488 0.09274581 0.08755051
  0.08601116 0.09601693 0.09082163 0.09082163]
 [0.22782374 0.23590533 0.23936887 0.23436598 0.23725226 0.23436598
  0.23359631 0.23398114 0.23994612 0.22724649]
 [0.08581874 0.08601116 0.08062344 0.07504329 0.08427939 0.07792957
  0.07234943 0.07465846 0.08139311 0.08466423]]
특성중요도 평균 [0.08876275 0.23438522 0.08027708]
표준편차 [0.00382333 0.00401363 0.00477012]


In [72]:
print('특성중요도 평균', result.importances_mean)
# 당도를 섞었을때 성능이 평균적으로 0.23만큼 떨였다는 뜻이니까
# 이 모델은 당도에 가장 크게 의존하고 있다. 

특성중요도 평균 [0.08876275 0.23438522 0.08027708]


In [ ]:
#테스트세트에서의 성능 최종확인, 정확더 얼마나 맞추었나. 
hgb.score(test_input, test_target)

0.8723076923076923

# XGBoost
- 사이킷런에서 제공하는 라이브러리가 아님

In [76]:
#%pip install XGBoost

In [78]:
from xgboost import XGBClassifier

xgb=XGBClassifier(tree_method='hist', random_state=42)
scores=cross_validate(xgb, train_input, train_target, return_train_score=True)
print(scores)

print('훈련점수', np.mean(scores['train_score']))
print('테스트점수',np.mean(scores['test_score']))

{'fit_time': array([0.15122628, 0.07922626, 0.06727362, 0.06635237, 0.06654978]), 'score_time': array([0.00484157, 0.00448012, 0.        , 0.        , 0.        ]), 'test_score': array([0.87115385, 0.88461538, 0.89990375, 0.86236766, 0.87391723]), 'train_score': array([0.95886457, 0.95718066, 0.95935546, 0.94516595, 0.96296296])}
훈련점수 0.9567059184812372
테스트점수 0.8783915747390243


In [80]:
#%pip install lightgbm

In [85]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(random_state=42)

scores = cross_validate(lgb, train_input, train_target, return_train_score=True)

print(np.mean(scores['train_score']))
print(np.mean(scores['test_score']))

0.935828414851749
0.8801251203079884
